# Project LAWA — BOD soft sensor models

**Goal:** estimate Biochemical Oxygen Demand (BOD, mg/L) at the 15 LLDA Laguna Lake stations from field-measurable readings, and compare Multiple Linear Regression (MLR), Random Forest and XGBoost (Chapter 3, Specific Objective 2).

**How to run**
1. Put `bod_class_dataset.csv` in Google Drive, in a folder named **Project LAWA** (My Drive › Project LAWA).
2. Runtime › Run all. Allow access to Google Drive when asked. A full run takes about 5–10 minutes (grid search).

**Contents**
1. Data overview · 2. Preprocessing (done in the workbook) · 3. Model inputs · 4. Train/test split · 5. Models and baselines · 6. Best model in detail · 7. DENR class check · 8. Year-by-year (rolling) test · 9. Random split comparison · 10. Benchmarks · 11. Final model for the dashboard

In [ ]:
# --- Libraries ---
import warnings, json, os
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
import sklearn, xgboost
from scipy.stats import ttest_rel
from sklearn.base import clone
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, confusion_matrix
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBRegressor

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})

SEED = 42                 # fixed random seed so results can be reproduced
TEST_START = "2024-01-01" # train on 2016-2023, test on 2024-2025
MAX_LAG_MONTHS = 3        # "last month's value" = previous record of the station, at most 3 months earlier
READINGS = ["DO", "pH", "Ammonia", "Nitrate", "Phosphate", "Temperature"]

print("scikit-learn", sklearn.__version__, "| xgboost", xgboost.__version__, "| pandas", pd.__version__)

In [ ]:
# --- Google Drive: where the CSV files are ---
# Put the CSV file(s) in a Drive folder named "Project LAWA" (My Drive > Project LAWA).
try:
    from google.colab import drive
    drive.mount("/content/drive")
    DATA_DIR = "/content/drive/MyDrive/Project LAWA"
except ImportError:                      # running outside Colab (e.g. on a laptop)
    DATA_DIR = os.environ.get("LAWA_DATA_DIR", ".")

OUT_DIR = os.path.join(DATA_DIR, "model_outputs")   # trained models are saved here
os.makedirs(OUT_DIR, exist_ok=True)
print("Reading data from:", DATA_DIR)
print("Files there:", sorted(f for f in os.listdir(DATA_DIR) if f.endswith(".csv")))

## 1. Data overview
One row per station per month, February 2016 to December 2025 (sheet *Preprocessing v4 (flagged)* of the LLDA workbook). Sources: LLDA compiled workbook (2016 to March 2023), LLDA quarterly report PDFs (April 2023 to December 2025, transcribed), LLDA temperature records.

In [ ]:
df = pd.read_csv(os.path.join(DATA_DIR, "bod_class_dataset.csv"))
df = df.rename(columns={"BOD_mgL": "BOD"})
df["Month"] = pd.to_datetime(df["Month"])
df["Outlier_flag"] = df["Outlier_flag"].fillna("")
TARGET_COL, LAST, FILLED = "BOD", "BOD_last", "BOD_imputed"

print(f"{len(df)} rows · {df.Station.nunique()} stations · {df.Month.min():%Y-%m} to {df.Month.max():%Y-%m}")
df.head()

In [ ]:
# Summary statistics of the readings and BOD
df[READINGS + ["BOD"]].describe().T.round(3)

In [ ]:
# DENR BOD classes (DAO 2016-08). Laguna Lake is Class C, so BOD above 7 mg/L is above standard.
BOD_LIMITS = [("A", 3), ("B", 5), ("C", 7), ("D", 15)]
CLASSES = ["A", "B", "C", "D", "Failed"]

def bod_class(v):
    for name, upper in BOD_LIMITS:
        if v <= upper:
            return name
    return "Failed"

counts = df["BOD"].map(bod_class).value_counts().reindex(CLASSES, fill_value=0)
print(pd.DataFrame({"rows": counts, "share": (counts / counts.sum()).round(3)}))
ax = counts.plot.bar(color=["#8ec5ea", "#dddddd", "#f2d64b", "#e06666", "#1a1a1a"], edgecolor="black", rot=0,
                     title="BOD classes in the dataset (most months are Class A)")
ax.set_ylabel("station-months"); plt.show()

In [ ]:
# Lake-wide monthly average BOD over time
monthly = df.groupby("Month")["BOD"].mean()
ax = monthly.plot(figsize=(11, 3.5), color="black", lw=1)
ax.axhline(7, color="red", ls="--", lw=1, label="Class C limit (7 mg/L)")
ax.set_ylabel("BOD (mg/L), mean of stations"); ax.set_title("Monthly lake-wide BOD"); ax.legend(); plt.show()

## 2. Preprocessing (already applied in the workbook)
1. **Rogue data:** physically impossible values treated as errors.
2. **Below detection limit:** "<x" stored as x/2 (same rule as the 2013–2023 workbook).
3. **Missing values:** linear interpolation within the station, gaps of up to 6 months only (`BOD_imputed = True`). Filled values are **never used as test targets**.
4. **Outliers:** values beyond ±3 SD are **flagged, not deleted** (`Outlier_flag`). The most extreme ones were checked against the official LLDA reports and are real (e.g. BOD 70 mg/L at Sta. Rosa, Sep 2017).
5. **TSS is not used:** no TSS data for 2016–2023.

In [ ]:
print("Filled (interpolated) BOD values:", int(df[FILLED].sum()))
print("Rows with an outlier flag:", int((df.Outlier_flag != "").sum()))
print("\nHighest BOD values (kept, verified):")
df.nlargest(5, "BOD")[["Month", "Station"] + READINGS + ["BOD", "Outlier_flag"]]

## 3. Model inputs
| Version | Inputs |
|---|---|
| A | the 6 readings: DO, pH, ammonia, nitrate, inorganic phosphate, temperature |
| B | A + station (one-hot) + month (sine/cosine) |
| C | B + last month's lab BOD at the same station (at most 3 months earlier) |

In [ ]:
# --- Model inputs ---
df = df.sort_values(["Station", "Month"]).reset_index(drop=True)

# last month's lab value of the target at the same station (only if at most 3 months earlier)
prev_month = df.groupby("Station")["Month"].shift(1)
gap = (df.Month.dt.year - prev_month.dt.year) * 12 + (df.Month.dt.month - prev_month.dt.month)
df[LAST] = df.groupby("Station")[TARGET_COL].shift(1).where(gap <= MAX_LAG_MONTHS)

# month as a cycle (December sits next to January)
df["month_sin"] = np.sin(2 * np.pi * df.Month.dt.month / 12)
df["month_cos"] = np.cos(2 * np.pi * df.Month.dt.month / 12)

# station as one-hot columns
stations = pd.get_dummies(df["Station"], prefix="stn", dtype=float)
STN = list(stations.columns)
df = pd.concat([df, stations], axis=1)

before = len(df)
df = df.dropna(subset=[LAST]).sort_values(["Month", "Station"]).reset_index(drop=True)
print(f"Rows: {before} -> {len(df)} (rows without a previous value within {MAX_LAG_MONTHS} months are dropped, "
      "so every version uses exactly the same rows)")

In [ ]:
VERSIONS = {
    "A: 6 readings": READINGS,
    "B: + station + month": READINGS + STN + ["month_sin", "month_cos"],
    "C: + last month's BOD": READINGS + STN + ["month_sin", "month_cos", LAST],
}
for v, cols in VERSIONS.items():
    print(f"{v}: {len(cols)} input columns")

## 4. Train/test split
**Time-based:** the models learn from 2016–2023 and are tested on 2024–2025, months they have never seen. Section 9 shows why the random 70:30 split in the manuscript gives over-optimistic scores.

In [ ]:
# --- Time-based split: train 2016-2023, test 2024-2025 ---
train = df[df.Month < TEST_START]
test = df[(df.Month >= TEST_START) & (~df[FILLED])]   # filled (interpolated) values are never used as test targets
print(f"Training rows: {len(train)} ({train.Month.min():%Y-%m} to {train.Month.max():%Y-%m})")
print(f"Test rows:     {len(test)} ({test.Month.min():%Y-%m} to {test.Month.max():%Y-%m}), measured values only")

## 5. Models and baselines
- **Baselines** every model must beat: *guess the training average* and *repeat last month's BOD*.
- **Tuning:** grid search with time-ordered 5-fold cross-validation on the training years only.
- **Metrics:** R², RMSE and MAE (mg/L), plus the share of estimates within ±1 mg/L.
- **Training cap at 15 mg/L:** for TRAINING only, BOD above 15 mg/L (the Class D limit; anything above is "Failed") is set to 15. Without it, the single real 70 mg/L month makes the tree models estimate about 60 mg/L for some ordinary months. Test values are never capped: the real 70 mg/L stays in the test data and the dataset.

In [ ]:
# --- Models compared (same three as the manuscript) and their tuning grids ---
MODELS = {
    "MLR": (make_pipeline(StandardScaler(), LinearRegression()), {}),
    "Random Forest": (RandomForestRegressor(random_state=SEED, n_jobs=-1),
                      {"n_estimators": [200, 500], "max_depth": [3, 5, 10, None], "min_samples_split": [2, 5, 10]}),
    "XGBoost": (XGBRegressor(random_state=SEED, n_jobs=-1, verbosity=0),
                {"learning_rate": [0.03, 0.1], "max_depth": [2, 3, 4], "n_estimators": [200, 500],
                 "subsample": [0.8, 1.0], "colsample_bytree": [0.8, 1.0]}),
}
CV = TimeSeriesSplit(n_splits=5)   # time-ordered: each fold validates on months AFTER its training months


def tune(model, grid, X, y):
    """Grid search on the TRAINING years only. Returns (fitted best model, best settings)."""
    if not grid:
        return clone(model).fit(X, y), {}
    gs = GridSearchCV(model, grid, cv=CV, scoring="neg_root_mean_squared_error", n_jobs=-1).fit(X, y)
    return gs.best_estimator_, gs.best_params_


def fold_rmse(model, X, y):
    """RMSE on each time-ordered CV fold of the training years (used for the paired t-test)."""
    out = []
    for tr, va in CV.split(X):
        m = clone(model).fit(X.iloc[tr], y.iloc[tr])
        out.append(np.sqrt(mean_squared_error(y.iloc[va], m.predict(X.iloc[va]))))
    return np.array(out)

In [ ]:
def score(y, p):
    e = np.abs(np.asarray(p) - np.asarray(y))
    return {"R2": r2_score(y, p), "RMSE": np.sqrt(mean_squared_error(y, p)), "MAE": mean_absolute_error(y, p),
            "within_1_mgL_%": 100 * (e <= 1).mean()}

CAP = 15                                  # training cap (mg/L), see the note above
y_tr, y_te = train["BOD"], test["BOD"]
y_fit = y_tr.clip(upper=CAP)              # target used for training
rows, preds, fitted, params, folds = [], {}, {}, {}, {}
rows.append({"version": "Baseline", "model": "Guess the average", **score(y_te, np.full(len(test), y_tr.mean()))})
rows.append({"version": "Baseline", "model": "Repeat last month", **score(y_te, test[LAST])})

for vname, cols in VERSIONS.items():
    for mname, (model, grid) in MODELS.items():
        m, p = tune(model, grid, train[cols], y_fit)
        key = (vname, mname)
        fitted[key], params[key], preds[key] = m, p, m.predict(test[cols])
        folds[key] = fold_rmse(m, train[cols], y_fit)
        rows.append({"version": vname, "model": mname, **score(y_te, preds[key])})
        print(f"done: {vname} / {mname} {p}")

results = pd.DataFrame(rows)
results.round(3)

In [ ]:
trained = results[results.version != "Baseline"]
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, metric, title in zip(axes, ["R2", "RMSE"], ["R² (higher is better)", "RMSE in mg/L (lower is better)"]):
    trained.pivot(index="version", columns="model", values=metric).loc[list(VERSIONS)].plot.bar(
        ax=ax, rot=0, color=["#9e9e9e", "#2e7d32", "#1565c0"])
    for _, b in results[results.version == "Baseline"].iterrows():
        ax.axhline(b[metric], ls="--", lw=1, color="red" if "last" in b.model else "black", label=f"baseline: {b.model}")
    ax.set_title(title); ax.set_xlabel(""); ax.legend(fontsize=7)
fig.suptitle("BOD models on 2024–2025 (never seen in training)"); plt.tight_layout(); plt.show()

## 6. Best model in detail
The best model is the one with the lowest test RMSE.

In [ ]:
best = trained.sort_values("RMSE").iloc[0]
BEST = (best.version, best.model)
best_cols = VERSIONS[best.version]
p_best = preds[BEST]
print(f"Best: {best.version} / {best.model}  ->  R² {best.R2:.3f}, RMSE {best.RMSE:.3f}, MAE {best.MAE:.3f}")
print("Settings:", params[BEST])

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
jitter = np.random.default_rng(SEED).uniform(-0.12, 0.12, len(y_te))   # BOD is mostly whole numbers
axes[0].scatter(y_te + jitter, p_best, s=12, alpha=0.5)
lim = [0, max(y_te.max(), p_best.max()) + 0.5]
axes[0].plot(lim, lim, "k--", lw=1); axes[0].set_xlim(lim); axes[0].set_ylim(lim)
axes[0].set_xlabel("Actual BOD (mg/L)"); axes[0].set_ylabel("Estimated BOD (mg/L)"); axes[0].set_title("Actual vs estimated")
axes[1].scatter(p_best, y_te - p_best, s=12, alpha=0.5); axes[1].axhline(0, color="k", lw=1)
axes[1].set_xlabel("Estimated BOD (mg/L)"); axes[1].set_ylabel("Error (actual − estimated)"); axes[1].set_title("Residuals")
plt.tight_layout(); plt.show()

ts = pd.DataFrame({"Month": test.Month.values, "actual": y_te.values, "estimate": p_best, "last month": test[LAST].values}).groupby("Month").mean()
ax = ts.plot(figsize=(11, 3.5), style=["k-o", "b-s", "r:"], ms=4)
ax.set_ylabel("BOD (mg/L), mean of stations"); ax.set_title("Monthly lake-wide BOD: actual vs estimated (test years)"); plt.show()

In [ ]:
# Accuracy per station (test years)
per_station = (pd.DataFrame({"Station": test.Station.values, "y": y_te.values, "p": p_best})
               .groupby("Station").apply(lambda g: pd.Series({"n": len(g), "RMSE": np.sqrt(mean_squared_error(g.y, g.p)),
                                                              "MAE": mean_absolute_error(g.y, g.p), "mean_BOD": g.y.mean()})))
per_station.sort_values("RMSE").round(2)

In [ ]:
# Which inputs matter most? Permutation importance: how much the test error grows when one input is shuffled.
imp = permutation_importance(fitted[BEST], test[best_cols], y_te, n_repeats=20, random_state=SEED,
                             scoring="neg_root_mean_squared_error")
importance = pd.Series(imp.importances_mean, index=best_cols)
grouped = pd.concat([importance[[c for c in best_cols if not c.startswith("stn_")]],
                     pd.Series({"station (all 15 together)": importance[[c for c in best_cols if c.startswith("stn_")]].sum()})])
ax = grouped.sort_values().plot.barh(figsize=(7, 4), color="#1565c0")
ax.set_xlabel("Increase in RMSE when shuffled (mg/L)"); ax.set_title(f"Input importance — {best.model}"); plt.show()

## 7. DENR class check
The estimate is converted to its DENR class and compared with the class of the actual lab value.

In [ ]:
actual_cls = y_te.map(bod_class)
est_cls = pd.Series(p_best, index=y_te.index).map(bod_class)
present = [c for c in CLASSES if c in set(actual_cls) | set(est_cls)]
cm = pd.DataFrame(confusion_matrix(actual_cls, est_cls, labels=present),
                  index=[f"actual {c}" for c in present], columns=[f"estimated {c}" for c in present])
print(cm, "\n")
worse = actual_cls != "A"
print(f"Class accuracy: {100 * (actual_cls == est_cls).mean():.1f}%  "
      f"(always guessing Class A would give {100 * (actual_cls == 'A').mean():.1f}%)")
print(f"Months worse than Class A that were flagged (B or worse): "
      f"{100 * (est_cls[worse] != 'A').mean():.1f}% of {int(worse.sum())}")

## 8. Year-by-year (rolling) test
Each year from 2018 to 2025 is predicted by models trained only on the years before it, with version C inputs and the settings found above.

In [ ]:
C = "C: + last month's BOD"
roll = []
for year in range(2018, 2026):
    tr = df[df.Month.dt.year < year]
    te = df[(df.Month.dt.year == year) & (~df[FILLED])]
    row = {"year": year, "n": len(te), "Repeat last month": r2_score(te.BOD, te[LAST])}
    for mname in MODELS:
        m = clone(fitted[(C, mname)]).fit(tr[VERSIONS[C]], tr.BOD.clip(upper=CAP))
        row[mname] = r2_score(te.BOD, m.predict(te[VERSIONS[C]]))
    roll.append(row)
roll = pd.DataFrame(roll).set_index("year")
print("R² per year (version C):")
print(roll.round(2))
print("\nMean R² over the years:")
print(roll.drop(columns="n").mean().round(3))

## 9. Comparison: random 70:30 split (manuscript method)
A random split puts neighbouring months of the same station in both training and test data, which makes scores look better than they will be on future months.

In [ ]:
rand_rows = []
tr, te = train_test_split(df, test_size=0.30, stratify=df.Station, random_state=SEED)   # stratified by station
te = te[~te[FILLED]]
for vname in VERSIONS:
    for mname in MODELS:
        m = clone(fitted[(vname, mname)]).fit(tr[VERSIONS[vname]], tr.BOD.clip(upper=CAP))
        rand_rows.append({"version": vname, "model": mname,
                          "R2 random split": r2_score(te.BOD, m.predict(te[VERSIONS[vname]])),
                          "R2 time split": results.set_index(["version", "model"]).loc[(vname, mname), "R2"]})
pd.DataFrame(rand_rows).round(3)

## 10. Benchmarks (Chapter 1)
- **Benchmark 1:** at least one model reaches R² ≥ 0.60 for BOD on the held-out test set.
- **Benchmark 3:** the best ensemble beats MLR by ≥ 15% lower RMSE **and** ≥ 0.10 higher R², with a paired t-test on the cross-validation fold errors.

In [ ]:
bench = []
for vname in VERSIONS:
    r = results[results.version == vname].set_index("model")
    ens = r.loc[["Random Forest", "XGBoost"]].sort_values("RMSE").index[0]
    rmse_cut = (r.loc["MLR", "RMSE"] - r.loc[ens, "RMSE"]) / r.loc["MLR", "RMSE"]
    r2_gain = r.loc[ens, "R2"] - r.loc["MLR", "R2"]
    t = ttest_rel(folds[(vname, "MLR")], folds[(vname, ens)])
    bench.append({"version": vname, "best R2": r.R2.max(), "Benchmark 1 met (R2 >= 0.60)": r.R2.max() >= 0.60,
                  "best ensemble": ens, "RMSE cut vs MLR %": 100 * rmse_cut, "R2 gain vs MLR": r2_gain,
                  "Benchmark 3 met": (rmse_cut >= 0.15) and (r2_gain >= 0.10), "paired t-test p": t.pvalue})
pd.DataFrame(bench).round(3)

## 11. Final model for the dashboard
The best model is refit on **all** years (2016–2025) so it uses the most recent data, then saved. Its accuracy is the one measured on 2024–2025 above.

In [ ]:
final = clone(fitted[BEST]).fit(df[best_cols], df["BOD"].clip(upper=CAP))
path = os.path.join(OUT_DIR, "bod_model.joblib")
joblib.dump({"model": final, "features": best_cols, "training_cap_mgL": CAP, "test_scores": best.to_dict(), "settings": params[BEST]}, path)
print("Saved:", path)

# Example: one station-month from the test years
example = test.iloc[[0]]
print("\nExample input:")
print(example[["Month", "Station"] + READINGS + [LAST]].to_string(index=False))
print(f"\nEstimated BOD: {final.predict(example[best_cols])[0]:.2f} mg/L   (actual lab value: {example.BOD.iloc[0]} mg/L)")
print("Note: this example was part of the final model's training data; the honest accuracy is the 2024–2025 test above.")

In [ ]:
# Summary
print(f"Best BOD model: {best.model} with inputs '{best.version}'")
print(f"Test 2024–2025: R² {best.R2:.3f}, RMSE {best.RMSE:.3f} mg/L, MAE {best.MAE:.3f} mg/L, within ±1 mg/L {best['within_1_mgL_%']:.0f}%")
base = results[results.version == "Baseline"].set_index("model")
print(f"Baselines: guess the average R² {base.loc['Guess the average', 'R2']:.3f}, repeat last month R² {base.loc['Repeat last month', 'R2']:.3f}")
print(f"Highest estimate the final model gives on all 2016–2025 inputs: {final.predict(df[best_cols]).max():.1f} mg/L (training cap {CAP} mg/L)")